Topic-RAG  (suitable for short documents and simple queries.)

In [4]:
import pandas as pd
import collections
import time
import faiss
import numpy

from sklearn.metrics.pairwise import cosine_similarity

from langchain_experimental.text_splitter import SemanticChunker
from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from sentence_transformers import SentenceTransformer , util
from umap import UMAP
from hdbscan import HDBSCAN
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance

In [3]:
!python --version

Python 3.10.14


In [4]:
import bertopic
bertopic.__version__

'0.16.3'

In [5]:
data_df= pd.read_pickle("../path_to_the_dataset.pkl")

In [ ]:
embeddings = data_df["sentence_embeddings"].to_list() # "document embeddings" column
contents = data_df["content_translated_processed_regex_cleaned"].to_list()  # 'cleaned content' column
print(len(embeddings), len(contents))


In [ ]:
contents, embeddings , years = numpy.array(contents), numpy.array(embeddings)
print(contents.shape, embeddings.shape)

#  BERTopic Modeling

In [12]:
# Extract vocab to be used in BERTopic
vocab_counter = collections.Counter()
tokenizer = CountVectorizer().build_tokenizer()

In [13]:
# Train model and reduce dimensionality of jina_bertopic_embeddings
umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, random_state=42, metric="cosine", verbose=True)

In [11]:
from bertopic.vectorizers import ClassTfidfTransformer

#  Cluster reduced embeddings
hdbscan_model = HDBSCAN(min_cluster_size=15, metric='euclidean', cluster_selection_method='eom', prediction_data=True,gen_min_span_tree=True)

#Tokenize topics
vectorizer_model = CountVectorizer(stop_words="english",ngram_range=(1,1))

# Create topic representation
ctfidf_model = ClassTfidfTransformer()

In [8]:
from sentence_transformers import SentenceTransformer 
embedding_model = SentenceTransformer("jinaai/jina-embeddings-v2-base-en", trust_remote_code=True)

In [9]:
# KeyBERT
keybert_model = KeyBERTInspired()

# MMR
mmr_model = MaximalMarginalRelevance(diversity=0.3)

# Representation models
representation_model = {
    "KeyBERT": keybert_model,
    "MMR": mmr_model,
  }

In [ ]:
start_time = time.time()

print("Start",start_time)

topic_model = BERTopic(
  # Pipeline models
  embedding_model=embedding_model,
  umap_model=umap_model,
  hdbscan_model=hdbscan_model,
  vectorizer_model=vectorizer_model,
  ctfidf_model=ctfidf_model, 
  representation_model=representation_model,
  top_n_words=10,
  verbose=True, calculate_probabilities=True,low_memory=True
)
topics,probs=topic_model.fit_transform(contents, embeddings)

end_time = time.time()
print("End",end_time)
execution_time = end_time - start_time

print("Execution time:", execution_time, "seconds")

In [38]:
import numpy as np

# Saving `probs` from the trained bertopic model - It is the topic probabilities array
np.save("../models/probs_RAGG.npy", probs)

In [ ]:
document_info_topic_model= topic_model.get_document_info(contents)
document_info_topic_model.head()

In [ ]:
topic_model.save("your_model_path.pkl",serialization="pickle",save_ctfidf=True, save_embedding_model=embedding_model)

# Load the saved Bertopic model

In [ ]:
from bertopic import BERTopic
bertopic_models= BERTopic.load("your_model_path.pkl")
bertopic_models.get_topic_info()

In [ ]:
bertopic_models.generate_topic_labels(10)

In [ ]:
# Load saved probabilites
import numpy as np

probs_load = np.load("../models/probs_RAGG.npy")
bertopic_models.visualize_distribution(probs_load[160])

In [ ]:
document_infor_topic_model= bertopic_models.get_document_info(contents)
# Merge the output with the metadata. Select the 'meta data' columns to merge from original data_df
data_to_merge=data_df[['uid','year','sentence_embeddings']]
merged_df = pd.concat([data_to_merge.reset_index(drop=True), document_infor_topic_model.reset_index(drop=True),],axis=1)


## Topic Embeddings

In [ ]:
topic_embeddings=bertopic_models.topic_embeddings_
topic_embeddings
# print(type(topic_embeddings))

# Creating FAISS index

In [22]:
embedding_dimension = embedding_model.get_sentence_embedding_dimension()
print(f"Embedding Dimension: {embedding_dimension}")

Embedding Dimension: 768


In [64]:
original_docs= merged_df['Document'].to_list()
len(original_docs)

4711

###  Topic-Document Vector Index 
- Created Topic-Document mapping. We arrange documents based on their topics. (separate index for each topic)

In [55]:
# document faiss index
embeddings_docs= merged_df['sentence_embeddings'].to_list()
embeddings_docs=numpy.array(embeddings_docs)
print(embeddings_docs.shape)

(4711, 768)


In [ ]:
topic_indices = merged_df['Topic'].values

topic_number_unique = merged_df['Topic'].unique()
topic_number_unique_count = len(topic_number_unique)

print("Unique topic numbers:", topic_number_unique)
print("Total topics:", topic_number_unique_count)

print ("-----------------------------------------")

print("Topic indices - an array mapping documents to topics:", topic_indices) # Eg. doc 1 mapped to topic 5, doc 2 mapped to topic 3, etc.

topic_indices = numpy.array(topic_indices)  # Shape: (num_documents,)
print ("Size of the topic Indices:", topic_indices.shape)

- We first create a dictionary to hold indices for each topic. The keys are the topic numbers and the values are the FAISS indices.
- We iterate over each topic , filter documents for each topic , create a FAISS index and store the index in the dictionary.
- Save the dictionary if needed.

In [ ]:
# Create a dictionary to hold indices for each topic
doc_indices = {} 

# Create FAISS index for each topic
for topic in numpy.unique(topic_indices):
    topic_docs = embeddings_docs[topic_indices == topic]

    print("\n")
    print(f"-------------------- Total number of documents: {topic_docs.shape[0]} associated with topic id: {topic}  --------------------\n")


    if topic_docs.shape[0] > 0:  
        doc_index = faiss.IndexFlatL2(topic_docs.shape[1])  # L2 distance
        doc_index.add(topic_docs.astype(numpy.float32))  
        doc_indices[topic] = doc_index  # Store the index in the dictionary

# Save the indices if needed
for topic, index in doc_indices.items():
    faiss.write_index(index, f'faiss_index/document_index_topic_{topic}.faiss')


In [59]:
doc_index # doc_index for the last topic

<faiss.swigfaiss.IndexFlatL2; proxy of <Swig Object of type 'faiss::IndexFlatL2 *' at 0x355134cf0> >

In [ ]:
doc_indices # Dictionary of all topic indices

In [61]:
new_doc_indices=doc_indices.copy()

In [ ]:
# Access the FAISS index
faiss_index = doc_indices[0]

# Get the total number of stored documents (the number of vectors in the index)
num_docs = faiss_index.ntotal

# If you want to see all document indices (not based on a query), you can simply retrieve the first N documents
all_doc_indices = [i for i in range(num_docs)]

print(f"All document indices for topic 1: {all_doc_indices}")



# Initialize quantized LLAMA 3.1 8b model - Generator Model to generate responses.

In [9]:
LLAMA_CPP_PATH = "../models/Meta-Llama-3.1-8B-Instruct-Q5_K_M.gguf" # from local directory
TOKENIZER_PATH= "meta-llama/Meta-Llama-3.1-8B-Instruct" # from Huggingface or local cache3

In [81]:
from transformers import AutoTokenizer
config = AutoTokenizer.from_pretrained(TOKENIZER_PATH, use_fast=True)
print(f"Maximum sequence length: {config.model_max_length}")

Maximum sequence length: 131072


In [116]:
from llama_cpp import Llama
from langchain_community.llms import LlamaCpp
from langchain.callbacks.manager import CallbackManager
from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler

callback_manager = CallbackManager([StreamingStdOutCallbackHandler()])

print("LOADING LLAMA.CPP MODEL...")
llama_model = Llama(
    model_path=LLAMA_CPP_PATH,
    n_gpu_layers=-1, # -1 put all models layers on the GPU
    max_tokens=8192, # output: chunk plus instructions (plus buffer for Llama-3 special tokens)
    n_ctx=100000, # input: chunk plus explanations (plus buffer for Llama-3 special tokens)
    f16_kv=True, # False means higher (=32bit) precision for key/value cache
    callback_manager=callback_manager,
    seed=42,
    #chat_format="chatml",
    verbose=False,
)
print("LLAMA.CPP MODEL LOADED.")

print("PREPARING LLAMA.CPP MODEL...")
from accelerate import Accelerator
accelerator = Accelerator()
llama_model = accelerator.prepare(llama_model)
print("LLAMA.CPP MODEL PREPARED.")

print("LOADING TOKENIZER...")
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_PATH, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
print("TOKENIZER LOADED.")

import torch
if torch.backends.mps.is_available():
    print("DEVICE:", round(torch.mps.driver_allocated_memory()/(1024.**3), 3), "GB ALLOCATED ON MPS")
elif torch.cuda.is_available():
    print("DEVICE:", round(torch.cuda.max_memory_reserved() /(1024.**3), 3), "GB ALLOCATED ON CUDA")    


LOADING LLAMA.CPP MODEL...
LLAMA.CPP MODEL LOADED.
PREPARING LLAMA.CPP MODEL...
LLAMA.CPP MODEL PREPARED.
LOADING TOKENIZER...
TOKENIZER LOADED.
DEVICE: 31.907 GB ALLOCATED ON MPS


#### Topic RAG (Our Method)

#### --------------------------------------------------------------------------------------------------------
- Generator Function (consist of 2 functions - 'generator prompt' and 'getAnswer')
    - "generator_prompt" function - a prompt for the generator model to follow this instruction and generate a response in this format.
    - "getAnswer" function - to generate a response based on the instruction and input text.
#### --------------------------------------------------------------------------------------------------------

In [121]:
def generator_prompt(instruction, input=None, uid=None, summary_size='concise'):
    return (
        "You are a skilled summarizer. Your task is to generate a {summary_size} summary of the provided context, "
        f"using the relevant document with UID {uid}. Focus solely on the information directly relevant to the instruction, ignoring irrelevant content. "
        "Ensure the summary is clear, concise, and presented in a logical order, avoiding any repetition of sentences and editorial comments. "
        "Do not repeat sentences to fill space; instead, provide a meaningful summary that aligns with the specified size.Provide a narrative summary that includes" "only the relevant information. Emphasize clarity and conciseness while covering all key points."
        "Avoid restating or repeating content, and ensure diversity in phrasing. Do not include any statements about the writing process in the beginning or end of" "the summary.\n\n"
        "### Instruction:\n"
        f"{instruction}\n\n"
        "### Input:\n"
        f"{input}\n\n"
        "### Generated Response:\n"
       
    ).format(summary_size=summary_size)

def getAnswer(INSTRUCTION, MAX_TOKENS=1000):
    
    combined_mapping = []
    print("\n")
    for topic in results["topic_info"]:
        print(f"Topic ID: {topic['topic_id']}, Probability: {topic['probability']}")
     # print(f"Retrieved Documents: {topic['retrieved_documents']}")

    
        selected_docs = topic['selected_documents']  # selected_docs = [(uid1, sim_score1), (uid2, sim_score2), ...]
        selected_doc_contents = topic['selected_doc_contents']
        
        
        # Create a mapping for each selected document
        for (uid, score), content in zip(selected_docs, selected_doc_contents):
            combined_mapping.append({
                "uid": uid,
                "score": score,
                "content": content,
                "topic_id": topic['topic_id'],  
            })
        for topic in results["topic_info"]:
            user_input = topic["user_input_value"]
            
        print("\n")  
        print(f"Selected Documents based on user input value '{(user_input)}' are as follows :")
        print("\n")
            # combined_mapping contains all selected documents with their UIDs, scores, topic IDs and content   
        for item in combined_mapping:
            print(f"UID: {item['uid']}, Score: {item['score']}, Topic ID: {item['topic_id']}, Content: {item['content']},\n")

    # Combine contents from selected documents in the combined mapping
    selected_doc_contents = [item['content'] for item in combined_mapping]  # Get the contents
    selected_doc_uid = [item['uid'] for item in combined_mapping]  # Get the UIDs
    input_text = '\n'.join(selected_doc_contents) # Combine the contents

    prompt = generator_prompt(INSTRUCTION, input=input_text , uid=selected_doc_uid,summary_size='concise')
    # inputs = tokenizer(prompt, return_tensors="pt")
    answer = llama_model(prompt, max_tokens=MAX_TOKENS,seed=42) # To generate response (according to the prompt) using llama model initialized above.
    # answer = model.generate(inputs.input_ids, max_new_tokens=MAX_TOKENS)
    
    print("\n-----------------------------------------------------------\n")
    print("Total number of retrieved documents",len(selected_doc_contents))
    print("\n-----------------------------------------------------------\n")
    
    print ("Original Documents",selected_doc_contents)
    print("\n---------------------------------\n")
    print("\n---------------------------------\n")
   
    
    return answer['choices'][0]['text']


- Retrieval Function (query_rag_system)
    - "query_rag_system" function - to query the RAG system and retrieve relevant documents. Metadata traceability. 


In [ ]:
import datetime

topic_ids = merged_df['Topic'].unique()

print("Total number of topics:", len(topic_ids))
print(f"Original topic IDs: {topic_ids}:")


def query_rag_system(query, k_topics, embedding_model, topic_model, new_doc_indices, docs_df): 
    
    
    # Step 1: Embed the query
    query_embedding = embedding_model.encode([query])  
    query_embedding = query_embedding.astype(np.float32)    
   

    # Step 2: Apply BERTopic modeling to get topic representations of the query
    topics, probs = topic_model.transform([query], query_embedding)  # Transform the query to get its topic
    
    print("Topics for the input query assigned by bertopic model :", topics)
    print("Probabilities for the input query assigned by bertopic model :", probs)
    
    # Find the highest probability
    highest_probability = np.max(probs[0]) # dominant topic probability
    
    # Set the threshold as 50% of the highest probability
    threshold = highest_probability * 0.5
    
    #>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>.
    ignored_topics = [i for i in range(len(probs[0])) if probs[0][i] < threshold]    
    # Print the ignored topics along with their probability values
    print("Ignored topics and their probabilities:")
    for topic in ignored_topics:
     print(f"Topic {topic}: Probability {probs[0][topic]:.4f}")
        
     #>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>.
    
    # Get indices of topics that meet or exceed the threshold
    filtered_topics = [i for i in range(len(probs[0])) if probs[0][i] >= threshold]   
    
    print(f"Filtered topics: {filtered_topics}")
    
    # Sort filtered topics based on probabilities
    top_query_topics_indices = sorted(filtered_topics, key=lambda idx: probs[0][idx], reverse=True)[:k_topics]
  

    # Check if the query maps to an existing topic
    if topics[0] != -1:
        print(f"The query maps to topic {topics[0]} with probability {probs[0][top_query_topics_indices]}")
    else:
        print("The query did not map to any existing topic and was considered an outlier.")

    print("Query Topic assigned by BERTopic model:", topics[0])
    print("Query Topic list based on Probs (list if more than one topic contributes to the input query ) :", topics) # list if more than one topic

    query_topic_ids = top_query_topics_indices
    print ("Documents to be retrieved based on the query topic ids:",query_topic_ids)
    top_probabilities = probs[0][top_query_topics_indices]
    print("Top probabilities:", top_probabilities)
    print("\n")
    
    topic_info = []

    for index, topic_idx in enumerate(query_topic_ids):  

        print(f"................................................Topic ID: {topic_idx} ................................................")

        #...............................................
        relevant_topic_id = int(topic_idx)
        print(f"Retrieving documents for topic ID {relevant_topic_id}")
        print(f"Processing Topic {index + 1}/{len(query_topic_ids)}: Topic ID {relevant_topic_id}")

        # topic_info = []
        retrieved_documents = {}
        retrieved_document_embeddings = {}
        retrieved_doc_contents = {}
        retrieved_doc_uid = {}
        selected_docs = {}
        selected_doc_contents = {}

       
        try:
            k_docss = new_doc_indices[relevant_topic_id].ntotal  # Total number of documents in the index
            print(f"Total number of documents in the index for topic ID {relevant_topic_id}: {k_docss}")
            distances, indices = new_doc_indices[relevant_topic_id].search(query_embedding, k_docss)  # Using FAISS to get the top k_docs
            retrieved_documents[relevant_topic_id] = indices[0].tolist()

            print(f"Initially Retrieved Documents for topic ID {relevant_topic_id}: {retrieved_documents[relevant_topic_id]}")           
            
            filtered_docs_df = docs_df[docs_df['Topic'] == relevant_topic_id]
            
            num_rows, num_columns = filtered_docs_df.shape
            print(f"Filtered DataFrame size: {num_rows} rows and {num_columns} columns")           
            
            print(f"Number of documents in the filtered DataFrame for topic {relevant_topic_id}: {num_rows}")
            
            retrieved_document_embeddings[relevant_topic_id] = filtered_docs_df['sentence_embeddings'].tolist()
            retrieved_doc_contents[relevant_topic_id] = filtered_docs_df['Document'].tolist()
            retrieved_doc_uid[relevant_topic_id] = filtered_docs_df['uid'].tolist()
                       
            print("\n")
            
            print(f"Size of retrieved_document_embeddings for Topic ID {relevant_topic_id}: {len(retrieved_document_embeddings[relevant_topic_id])}")


            # Calculate cosine similarity between the query embedding and retrieved document embeddings
            cosine_similarities = cosine_similarity(query_embedding.reshape(1, -1), np.array(retrieved_document_embeddings[relevant_topic_id])).flatten()

            
            document_scores = [
                (retrieved_doc_uid[relevant_topic_id][i], cosine_similarities[i])
                for i in range(len(cosine_similarities))
            ]
            # Sort the documents by cosine similarity in descending order
            sorted_document_scores = sorted(document_scores, key=lambda x: x[1], reverse=True)

            # print(f"\nCosine Similarity Scores for Topic ID {relevant_topic_id} as follows : \n ")
            # 
            # for rank, (doc_uid, sim) in enumerate(sorted_document_scores):
            #     print(f"Rank {rank + 1}: Document UID: {doc_uid}, Cosine Similarity: {sim}")               

            #..............................
            # Prepare the top 15 cosine similarities for the input prompt
            top_n = 15
            top_scores = sorted_document_scores[:top_n]  # Get the top n scores
            top_scores_str = "\n".join([f"Rank {rank + 1}: UID: {doc_uid}, Similarity: {sim:.4f}" 
                                         for rank, (doc_uid, sim) in enumerate(top_scores)])                                                    

             # Ask the user how many documents to retrieve from the same cluster (here it is topic id )
            user_input = int(input(f"How many documents would you like to select in topic {relevant_topic_id} "
                                   f"for further processing from the {len(cosine_similarities)} documents?\n Query :{[query]} \n\n"
                                   f"Top {top_n} Cosine Similarities:\n{top_scores_str}\n"))

            # Select the top user_input documents based on sorted cosine similarities
            selected_docs[relevant_topic_id] = sorted_document_scores[:user_input]
            print("\n")
            # Print the selected documents with their topic information
            print(f"Selected Documents with Topic Information for Topic ID {relevant_topic_id}: {selected_docs[relevant_topic_id]}")

            # Get the actual content of the selected documents
            selected_doc_contents[relevant_topic_id] = [retrieved_doc_contents[relevant_topic_id][
                retrieved_doc_uid[relevant_topic_id].index(doc_uid)
            ] for doc_uid, _ in selected_docs[relevant_topic_id]]

            # print(f"Selected Document Contents for Topic ID {relevant_topic_id}: {selected_doc_contents[relevant_topic_id]}")

            topic_details = {
                "topic_id": relevant_topic_id,
                "probability": probs[0][relevant_topic_id] if relevant_topic_id < len(probs[0]) else None,
                "retrieved_documents": retrieved_documents.get(relevant_topic_id, []),
                "retrieved_embeddings": retrieved_document_embeddings.get(relevant_topic_id, []),
                "retrieved_doc_contents": retrieved_doc_contents.get(relevant_topic_id, []),
                "retrieved_doc_uid": retrieved_doc_uid.get(relevant_topic_id, []),
                "selected_documents": selected_docs[relevant_topic_id],
                "selected_doc_contents": selected_doc_contents[relevant_topic_id],
                "user_input_value": user_input,
            }
            topic_info.append(topic_details)

        except KeyError:
            print(f"Topic ID {relevant_topic_id} not found in new_doc_indices.")

    return {
        "query": query,
        "query_embedding": query_embedding,
        "topic_info": topic_info
    }


### Main Function 

In [ ]:
start_time = datetime.datetime.now()

# ________________________________________________________________________________________________________________________________

query="Which articles strongly advocate nuclear power. What are the arguments they put forward? Which articles strongly oppose nuclear power. What are the arguments they put forward? Do arguments change over time? ?"
# _____________________________________________________________________________________________________________________


k_topics = 5  # Number of top topics to retrieve (query), you can set this according to the complexity of the query or user-preferences

# -------------------------------------Retrieval------------------------------------------------------------------
retrieval_start_time = datetime.datetime.now()

# Retrieval based on topic representation of the query and then by query embedding.
results = query_rag_system(query, k_topics, embedding_model, bertopic_models, new_doc_indices, merged_df)

retrieval_end_time = datetime.datetime.now()

elapsed_time_retrieval = retrieval_end_time - retrieval_start_time


print ("........................Retrieval completed..................... \n ")

print("Elapsed time for retrieval",elapsed_time_retrieval)

print("----------------------------------------------------------------------------------")

# -------------------------------------Generation------------------------------------------------------------------
print("\n")
print( "........................Generating Response - Starting......................")

generation_time_start = datetime.datetime.now()

response = getAnswer(INSTRUCTION=f"Explain the details of {query}")
print("__________________________________Generated Response:________________________--")
print(response)
generation_time_end = datetime.datetime.now()

generated_response_time = generation_time_end - generation_time_start

print("Elapsed time for generation",generated_response_time)

print("******************************************************")

end_time =datetime.datetime.now()

Total_time = end_time - start_time

print("Total Elapsed time ( both retrieval and generation) ",Total_time)

#### Base RAG Implementation - This is our baseline 

In [ ]:
uid = data_df['uid'].to_list()

In [ ]:
import faiss
import numpy as np

#  embeddings_docs contains the embeddings for all documents
# embeddings_docs should be a NumPy array of shape (num_documents, embedding_dim)
doc_index_naive = faiss.IndexFlatL2(embeddings_docs.shape[1])  # L2 distance, shape[1] gives embedding dimension

doc_index_naive.add(embeddings_docs.astype(np.float32))  # Ensure embeddings are float32

faiss.write_index(doc_index_naive, 'faiss_index_naive/document_index.faiss')

print(f"Total number of documents added to FAISS index: {doc_index_naive.ntotal}")

In [ ]:
import datetime


dimension = embedding_dimension  

# Naive retrieval function
def retrieve_documents(query, k):
   
    query_embedding = embedding_model.encode([query]).astype(np.float32)
    
    # Search in the base FAISS index (without topic information)
    distances, indices = doc_index_naive.search(query_embedding, k)
    
    print("Docuemnet Indices",indices[0])
    print("Indices retrieved from FAISS:", indices[0])
    print("Length of documents:", len(documents))
    
    # Retrieve documents and compute cosine similarity
    valid_indices = [int(i) for i in indices[0].tolist()]
    print("Valid indices:", valid_indices)

       # Check if all indices are valid
    if all(0 <= i < len(documents) for i in valid_indices):
        retrieved_docs = [documents[i] for i in valid_indices]
        retrieved_uid = [uid[i] for i in valid_indices] 
        cosine_similarities = cosine_similarity(query_embedding.reshape(1, -1), embeddings_docs[valid_indices])
        
      
        # Sort the results by cosine similarity
        sorted_docs = sorted(zip(retrieved_docs, retrieved_uid, cosine_similarities[0]), key=lambda x: x[2], reverse=True)
        return sorted_docs
    else:
        print("Invalid indices found:", [i for i in valid_indices if not (0 <= i < len(documents))])
        return []

def prompt_naive(instruction, input=None,uid=None , summary_size='concise'):
    
       return (
        "You are a skilled summarizer. Your task is to generate a {summary_size} summary of the provided context, "
        f"using the relevant document with UID {uid}. Focus solely on the information directly relevant to the instruction, ignoring irrelevant content. "
        "Ensure the summary is clear, concise, and presented in a logical order, avoiding any repetition of sentences and editorial comments. "
        "Do not repeat sentences to fill space; instead, provide a meaningful summary that aligns with the specified size.Provide a narrative summary that includes" "only the relevant information. Emphasize clarity and conciseness while covering all key points."
        "Avoid restating or repeating content, and ensure diversity in phrasing. Do not include any statements about the writing process in the beginning or end of" "the summary.\n\n"
        "### Instruction:\n"
        f"{instruction}\n\n"
        "### Input:\n"
        f"{input}\n\n"
        "### Generated Response:\n"
       
    ).format(summary_size=summary_size)
  
    
# Step 3: Generate answer from Llama based on retrieved documents
def get_answer_from_llama(query, instruction, k):
    # Retrieve top-k documents
    r_start_time = datetime.datetime.now()
    retrieved_docs = retrieve_documents(query, k)

   
    input_text = "\n".join([doc for doc, uid, sim in retrieved_docs])
    r_end_time = datetime.datetime.now()
    total_r_time = r_end_time - r_start_time
    print("Retrieval time NAIVE",total_r_time)
    # Prepare the prompt for the generator model
    generator_input = prompt_naive(instruction, input=input_text, summary_size='concise')
    
    # Generate response using Llama
    answer = llama_model(generator_input, max_tokens=500)  # Adjust max_tokens as per your requirements
    
    print("\n-----------------------------------------------------------\n")
    print(f"Query: {query}")
    print(f"Top {k} Retrieved Documents (sorted by cosine similarity):")
    for rank, (doc, uid, sim) in enumerate(retrieved_docs, 1):
        print(f"Rank {rank}: UID: {uid}  | Document: {doc[:500]}... | Cosine Similarity: {sim:.4f}")
    return answer['choices'][0]['text'],retrieved_docs


# Test the naive RAG system
start_time = datetime.datetime.now()
# query = "Pro and cons of nuclear power plants?"
# query ='How many signatures were handed over to the government councils in Baselstadt, Baselland, Aargau, Solothurn, and Bern ?'
query = "was there a fire in moscow on thursday night in the train station?"
instruction = f"Explain the {query}"
response_naive,retrieved_docs= get_answer_from_llama(query, instruction, k=1)
end_time =datetime.datetime.now()

Total_time = end_time - start_time


# Output the generated response
print("Generated Response:")
print(response_naive)

print(".....................")
print("Elapsed time",Total_time)


#### Topic-RAG also support Multiple Queries inference , you like below.

In [ ]:
def generator_prompt_test(instruction, input=None, uid=None, summary_size='concise'):
    return (
        "You are a skilled answer generator. Your task is to generate a {summary_size} summary of the provided context, "
        f"using the relevant document with UID {uid}. Focus solely on the information directly relevant to the instruction, ignoring irrelevant content. "
        "Ensure the summary is clear, concise, and presented in a logical order, avoiding any repetition of sentences and editorial comments. "
        "Do not repeat sentences to fill space; instead, provide a meaningful summary that aligns with the specified size.Emphasize clarity and conciseness while covering the key points."
        "Avoid restating or repeating content, and ensure diversity in phrasing. Do not include any statements about the writing process in the beginning or end of" "the answer?.\n\n"
        "### Instruction:\n"
        f"{instruction}\n\n"
        "### Input:\n"
        f"{input}\n\n"
        "### Generated Response:\n"

    ).format(summary_size=summary_size)

def getAnswer_test(INSTRUCTION, MAX_TOKENS=100):

    combined_mapping = []

    for topic in results["topic_info"]:
     print(f"Topic ID: {topic['topic_id']}, Probability: {topic['probability']}")
     # print(f"Retrieved Documents: {topic['retrieved_documents']}")


    selected_docs = topic['selected_documents']  # selected_docs = [(uid1, sim_score1), (uid2, sim_score2), ...]
    selected_doc_contents = topic['selected_doc_contents']

    # Create a mapping for each selected document
    for (uid, score), content in zip(selected_docs, selected_doc_contents):
        combined_mapping.append({
            "uid": uid,
            "score": score,
            "content": content,
            "topic_id": topic['topic_id'],
        })

        # combined_mapping contains all selected documents with their UIDs, scores, topic IDs and content
    for item in combined_mapping:
        print(f"UID: {item['uid']}, Score: {item['score']}, Topic ID: {item['topic_id']}, Content: {item['content']},\n")

    # Combine contents from selected documents in the combined mapping
    selected_doc_contents = [item['content'] for item in combined_mapping]  # Get the contents
    selected_doc_uid = [item['uid'] for item in combined_mapping]  # Get the UIDs
    input_text = '\n'.join(selected_doc_contents) # Combine the contents

    prompt = generator_prompt_test(INSTRUCTION, input=input_text , uid=selected_doc_uid,summary_size='concise')
    inputs = tokenizer(prompt, return_tensors="pt")
    answer = llama_model(prompt, max_tokens=MAX_TOKENS,seed=42)
    # answer = model.generate(inputs.input_ids, max_new_tokens=MAX_TOKENS)

    print("\n-----------------------------------------------------------\n")
    print("Total number of retrieved documents",len(selected_doc_contents))
    print("\n-----------------------------------------------------------\n")

    print ("Original Documents",selected_doc_contents)
    print("\n---------------------------------\n")
    print("\n---------------------------------\n")


    return answer['choices'][0]['text']


# Assuming topic_model and embedding_model are already instantiated
topic_ids = merged_df['Topic'].unique()

print(f"Topic IDs: {topic_ids}:")

def query_rag_system(query, k_topics, embedding_model, topic_model, new_doc_indices, docs_df):
    # Step 1: Embed the query
    query_embedding = embedding_model.encode([query])  # Embed the query as a batch
    query_embedding = query_embedding.astype(np.float32)  # Ensure the correct data type

    print("Total number of topics:", len(topic_ids))
    print(f"Original topic IDs: {topic_ids}:")

    # Step 2: Apply BERTopic modeling to get topic representations
    topics, probs = topic_model.transform([query], query_embedding)  # Transform the query to get its topic

    print("Topics for the input query assigned by bertopic model :", topics)

    # Find the highest probability
    highest_probability = np.max(probs[0])

    # Set the threshold as 50% of the highest probability
    threshold = highest_probability * 0.5

    # Get indices of topics that meet or exceed the threshold
    filtered_topics = [i for i in range(len(probs[0])) if probs[0][i] >= threshold]

    # Sort filtered topics based on probabilities
    top_query_topics_indices = sorted(filtered_topics, key=lambda idx: probs[0][idx], reverse=True)[:k_topics]

    print(f"Top indices based on probabilities from BERTopic model: ", top_query_topics_indices)

    # Check if the query maps to an existing topic
    if topics[0] != -1:
        print(f"The query maps to topic {topics[0]} with probability {probs[0]}")
    else:
        print("The query did not map to any existing topic and was considered an outlier.")

    print("Query Topic assigned by BERTopic model:", topics[0])

    query_topic_ids = top_query_topics_indices
    print(f"Top {k_topics} topic IDs:", query_topic_ids)
    top_probabilities = probs[0][top_query_topics_indices]
    print("Top probabilities:", top_probabilities)

    topic_info = []
    # retrieved_documents = {}
    # retrieved_document_embeddings = {}
    # retrieved_doc_contents = {}
    # retrieved_doc_uid = {}

    for index, topic_idx in enumerate(query_topic_ids):

        print(f"................................................Topic ID: {topic_idx} ................................................")

        #...............................................
        relevant_topic_id = int(topic_idx)
        print(f"Retrieving documents for topic ID {relevant_topic_id}")
        print(f"Processing Topic {index + 1}/{len(query_topic_ids)}: Topic ID {relevant_topic_id}")

        # topic_info = []
        retrieved_documents = {}
        retrieved_document_embeddings = {}
        retrieved_doc_contents = {}
        retrieved_doc_uid = {}
        selected_docs = {}
        selected_doc_contents = {}

        # Retrieve the top k documents for this topic using the corresponding FAISS index
        try:
            k_docss = new_doc_indices[relevant_topic_id].ntotal  # Total number of documents in the index
            print(f"Total number of documents in the index for topic ID {relevant_topic_id}: {k_docss}")
            distances, indices = new_doc_indices[relevant_topic_id].search(query_embedding, k_docss)  # Using FAISS to get the top k_docs
            retrieved_documents[relevant_topic_id] = indices[0].tolist()

            print(f"Initially Retrieved Documents for topic ID {relevant_topic_id}: {retrieved_documents[relevant_topic_id]}")

            # Filter docs_df to include only the rows corresponding to the relevant topic
            filtered_docs_df = docs_df[docs_df['Topic'] == relevant_topic_id]
            # Check the size of the filtered DataFrame (number of rows and columns)
            num_rows, num_columns = filtered_docs_df.shape
            print(f"Filtered DataFrame size: {num_rows} rows and {num_columns} columns")
            
            # Optionally, you can just check the number of rows (documents)
            print(f"Number of documents in the filtered DataFrame for topic {relevant_topic_id}: {num_rows}")
            
            retrieved_document_embeddings[relevant_topic_id] = filtered_docs_df['sentence_embeddings'].tolist()
            retrieved_doc_contents[relevant_topic_id] = filtered_docs_df['Document'].tolist()
            retrieved_doc_uid[relevant_topic_id] = filtered_docs_df['uid'].tolist()




            # Calculate cosine similarity between the query embedding and retrieved document embeddings
            cosine_similarities = cosine_similarity(query_embedding.reshape(1, -1), np.array(retrieved_document_embeddings[relevant_topic_id])).flatten()

            # Create a list of tuples (document_uid, cosine_similarity)
            document_scores = [
                (retrieved_doc_uid[relevant_topic_id][i], cosine_similarities[i])
                for i in range(len(cosine_similarities))
            ]
            # Sort the documents by cosine similarity in descending order
            sorted_document_scores = sorted(document_scores, key=lambda x: x[1], reverse=True)

            # Print sorted results topic-wise
            # Display the similarity values to the user
            print(f"\nCosine Similarity Scores for Topic ID {relevant_topic_id}:")
            for rank, (doc_uid, sim) in enumerate(sorted_document_scores):
                print(f"Rank {rank + 1}: Document UID: {doc_uid}, Cosine Similarity: {sim}")

            #..............................
            # Prepare the top 10 cosine similarities for the input prompt
            top_n = 25
            top_scores = sorted_document_scores[:top_n]  # Get the top 10 scores
            top_scores_str = "\n".join([f"Rank {rank + 1}: UID: {doc_uid}, Similarity: {sim:.4f}"
                                         for rank, (doc_uid, sim) in enumerate(top_scores)])



             # Ask the user how many documents to retrieve from the same cluster
            user_input = int(input(f"How many documents would you like to select in topic {relevant_topic_id} "
                                   f"for further processing from the {len(cosine_similarities)} documents?\n\n"
                                   f"Top {top_n} Cosine Similarities:\n{top_scores_str}\n"))

            # Select the top user_input documents based on sorted cosine similarities
            selected_docs[relevant_topic_id] = sorted_document_scores[:user_input]

            # Print the selected documents with their topic information
            print(f"Selected Documents with Topic Information for Topic ID {relevant_topic_id}: {selected_docs[relevant_topic_id]}")

            # Get the actual content of the selected documents
            selected_doc_contents[relevant_topic_id] = [retrieved_doc_contents[relevant_topic_id][
                retrieved_doc_uid[relevant_topic_id].index(doc_uid)
            ] for doc_uid, _ in selected_docs[relevant_topic_id]]

            print(f"Selected Document Contents for Topic ID {relevant_topic_id}: {selected_doc_contents[relevant_topic_id]}")

            topic_details = {
                "topic_id": relevant_topic_id,
                "probability": probs[0][relevant_topic_id] if relevant_topic_id < len(probs[0]) else None,
                "retrieved_documents": retrieved_documents.get(relevant_topic_id, []),
                "retrieved_embeddings": retrieved_document_embeddings.get(relevant_topic_id, []),
                "retrieved_doc_contents": retrieved_doc_contents.get(relevant_topic_id, []),
                "retrieved_doc_uid": retrieved_doc_uid.get(relevant_topic_id, []),
                "selected_documents": selected_docs[relevant_topic_id],
                "selected_doc_contents": selected_doc_contents[relevant_topic_id],
            }
            topic_info.append(topic_details)

        except KeyError:
            print(f"Topic ID {relevant_topic_id} not found in new_doc_indices.")

    return {
        "query": query,
        "query_embedding": query_embedding,
        "topic_info": topic_info
    }


# Step 7: Test the function

queries = ["Was there a fire in Moscow on Thursday night in the train station?", "Explain three mile island accident",
    "How many signatures were handed over to the government councils in Baselstadt, Baselland, Aargau, Solothurn, and Bern?,"
    "What was the purpose of the petition handed over to the government councils?",  "What was the symbolic gesture made by the delegates during the meeting in Aarau? ",  "Why did the delegates present the basket of agricultural products to Fricker and Guggisberg" , "What might be the opinion of the delegates regarding the construction of the Kaiseraugst nuclear power plant?","What was the decision made by the Administrative Court of the Canton of Aargau regarding the Kaiseraugst Aarau nuclear power plant?","What decision did the city of Basel take regarding the nuclear power plant in Kaiseraugst?", "How does the decision of the Basel city government impact the ongoing dispute over the nuclear power plant in Kaiseraugst?", "Why did Baselcity change its stance on appealing the nuclear power plant decision?","What might be the potential implications of Baselcity's decision for the future of nuclear energy in Switzerland?"
]

k_topics = 5  # Number of top topics/documents to retrieve

all_results = []  # List to store results for all queries

# Iterate through the list of queries
for query in queries:
    print(f"Processing query: {query}")
    results = query_rag_system(query, k_topics, embedding_model, bertopic_models,  new_doc_indices, merged_df)

    # Optionally, you can generate responses for each query here
    print("Generating Response - Started........")
    response = getAnswer_test(INSTRUCTION=f"Explain the details of {query}")

    # Store the result along with the response
    all_results.append({
        "query": results["query"],
        "topic_info": results["topic_info"],
        "response": response
    })


# Step 8: Print the results for each query
print("...............................OUTPUT................................................")
for result in all_results:
    print("Query:", result["query"])
    for topic in result["topic_info"]:
        print(f"Topic ID: {topic['topic_id']}, Probability: {topic['probability']}")
        print(f"Selected Documents: {topic['selected_documents']}")
        print(f"Selected Document Contents: {topic['selected_doc_contents']}")
    print("Response:", result["response"])
    print("******************************************************")

In [ ]:
for result in all_results:
    print("Query:", result["query"])
    for topic in result["topic_info"]:
        # print(f"Topic ID: {topic['topic_id']}, Probability: {topic['probability']}")
        # print(f"Selected Documents: {topic['selected_documents']}")
        # print(f"Selected Document Contents: {topic['selected_doc_contents']}")
        print("Response:", result["response"])
    print("******************************************************")